# Lecture 5 — Expectations and Transition Dynamics

**Computational Methods for Heterogeneous-Agent Macro**

Jeffrey Sun


## 1. Setup

We load `HouseholdStages` together with plotting and `Accessors` (for
the `@set` macro used in comparative statics).


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages, Plots, Printf, Accessors

## 2. The Aiyagari model in `HouseholdStages`

Same three-stage household problem as L04, plus a Cobb-Douglas firm
and equilibrium $\bar K = \bar K^{\mathrm{supplied}}$. TFP `A` enters
through `aiyagari_prices`; the household chain's env carries only
prices `(r, w)`.

### 2.1 Parameters


In [ ]:
@kwdef struct AiyagariParams
    β::Float64 = 0.96
    σ::Float64 = 1.5
    α::Float64 = 0.36
    δ::Float64 = 0.08
    L::Float64 = 1.0
    A::Float64 = 1.0
    z_grid::Vector{Float64} = [0.6, 1.0, 1.4]
    P_z::Matrix{Float64}    = [0.7 0.2 0.1;
                               0.2 0.6 0.2;
                               0.1 0.2 0.7]
    N_w::Int       = 400
    w_min::Float64 = 0.0
    w_max::Float64 = 100.0
end

### 2.2 Layout and household chain

Three stages, in time order:

1. **Markov productivity shock** — `MarkovStage` along the `:z` axis.
2. **Receive income** — `WealthChangeStage`, with the
   period-budget identity $w' = (1+r)\,w + w\cdot z$.
3. **Consumption-savings choice** — `ConsumptionSavingsStage` with
   CRRA utility.

Compose them with `∘` (time order: leftmost runs first), then
`define_moments!` attaches the aggregate-wealth integral that the
outer tatonnement reads.

   $w' = (1+r)\,w + w\cdot z$。


In [ ]:
aiyagari_layout(p) = StateLayout(
    StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
    StateAxis(:z, p.z_grid),
)

_u_crra(c, ::Val{1}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ) = c < 0 ? -Inf : _u_crra(c, valσ)

function aiyagari_household(p::AiyagariParams)
    layout = aiyagari_layout(p)

    # Stage 1 — Markov productivity shock
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)

    # Stage 2 — Receive income
    income = WealthChangeStage(layout; wealth_post=(cell; env) -> (1 + env.r) * cell.wealth + env.w * cell.z)

    # Stage 3 — Consumption-savings
    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.σ)), monotone_search=:divide_conquer)

    # Compose in time order — leftmost runs first
    hh = z_shock ∘ income ∘ savings

    # Attach the K_supplied moment (aggregate wealth integral at end of chain)
    return define_moments!(hh; K_supplied=at_end(integrand=:wealth, reduce=sum))
end

# Cobb-Douglas factor prices; A is read from p, so the env stays (r, w).
function aiyagari_prices(K, p::AiyagariParams)
    (; α, δ, L, A) = p
    return (; r=A*α*(K/L)^(α-1) - δ, w=A*(1-α)*(K/L)^α)
end

### 2.3 What does `hh` look like?

A bundled `ChainStage` carries one Spec (pure configuration: layouts,
transition matrix, closures, attached moments) and one Buffer
(per-call state: kernels, scratch arrays, warm-start V/Λ, kernel
cache). Users never touch the Buffer directly — they just pass `hh`
into the package's helpers.


In [ ]:
hh = aiyagari_household(AiyagariParams())
dump(hh; maxdepth=1)

### 2.4 Single-env probe

The user-facing surface for "solve at a single env" is
`solve_steady_state_given_env!(hh, env)`. It runs the V backward iteration and
the Λ forward iteration, returns copies of `V`, `Λ`, and the
defined `moments` — and incidentally warm-starts the chain's buffer
for the next call.

Build the env with `make_env(hh; ...)`. The helper validates field
names against the chain's schema and raises a clear error if any
required key is missing.


In [ ]:
p   = AiyagariParams()
hh  = aiyagari_household(p)
env = make_env(hh; aiyagari_prices(5.0, p)...)
res = solve_steady_state_given_env!(hh, env)

@printf "K_supplied = %.4f  (K_guess = 5.0)\n" res.moments.K_supplied
@printf "VFI iters: %d   Λ iters: %d\n" res.history.vfi_iters res.history.lambda_iters

### 2.5 Steady state via tatonnement on K

Outer loop: guess $K$, get $K^{\mathrm{supplied}}$ from the household
block, damped-update $K$, repeat. The chain's buffer warm-starts each
call automatically, so the inner solves get cheaper as the outer
iteration progresses.


In [ ]:
function aiyagari_steady_state(p::AiyagariParams; K_init=5.0, update_speed=0.01)
    hh = aiyagari_household(p)
    K  = K_init
    local V, Λ
    history = Float64[]
    for it in 1:500
        env = make_env(hh; aiyagari_prices(K, p)...)
        (;V, Λ, moments) = solve_steady_state_given_env!(hh, env)
        K_S = moments.K_supplied
        err = abs(K_S - K) / K
        push!(history, err)
        err ≤ 2e-2 && return (; K, V, Λ, hh, history, iters=it)
        K += update_speed * (K_S - K)
    end
    error("aiyagari_steady_state: did not converge")
end

ss = aiyagari_steady_state(p)
@printf "K_ss = %.4f, r = %.4f, w = %.4f  (in %d outer iters)\n" ss.K aiyagari_prices(ss.K, p).r aiyagari_prices(ss.K, p).w ss.iters

### 2.6 Comparative statics

Permanent 5% positive TFP shock. We swap `p.A` with `@set` and
re-run the tatonnement starting from the old SS (a good warm start).


In [ ]:
ss_old = aiyagari_steady_state(p)
p_new  = @set p.A = 1.05
ss_new = aiyagari_steady_state(p_new; K_init=ss_old.K * 1.05)

@printf "ΔK = %+0.4f  (%.2f%%)\n" (ss_new.K - ss_old.K) 100*(ss_new.K/ss_old.K - 1)
@printf "old K_ss = %.4f   new K_ss = %.4f\n" ss_old.K ss_new.K

## 3. MIT shock — perfect-foresight transition

**Given.** The exogenous path $\{A_t\}_{t=1}^T$, the household chain
`hh`, and the firm $(\alpha, \delta, L)$.

**Find.** Sequences $\{K_t\}, \{V_t\}, \{\Lambda_t\}$ such that

- $V_{T+1} = V_{\mathrm{ss\_new}}$ (terminal condition),
- $\Lambda_1 = \Lambda_{\mathrm{ss\_old}}$ (initial condition),
- $V_t = \texttt{backward!}(V_{t+1}, \mathrm{env}_t)$ for $t = T, T{-}1, \ldots, 1$,
- $\Lambda_{t+1} = \texttt{forward!}(\Lambda_t, V_{t+1}, \mathrm{env}_t)$ for $t = 1, 2, \ldots, T$,
- $K_t = \int b\,\mathrm{d}\Lambda_{t+1} = K_t^{\mathrm{supplied}}$ (market clears every period).

**Algorithm.** Guess $\{K_t\}$, sweep $V$ backward from the terminal
condition, sweep $\Lambda$ forward from the initial condition, read
off $K_t^{\mathrm{supplied}}$, damped-update $\{K_t\}$.

- $V_t = \texttt{backward!}(V_{t+1}, \mathrm{env}_t)$，$t = T, T{-}1, \ldots, 1$，
- $\Lambda_{t+1} = \texttt{forward!}(\Lambda_t, V_{t+1}, \mathrm{env}_t)$，$t = 1, 2, \ldots, T$，

### 3.1 Clean version — using `solve_transition_given_env_path!`

`solve_transition_given_env_path!` owns the per-period buffer allocation, the
backward and forward sweeps with kernel-cache-correct re-seating,
and the per-period moment computation. The outer loop just supplies
the env path and the boundary conditions.


In [ ]:
function mit_shock_transition(p::AiyagariParams; A_new=1.05, T=100, update_speed=0.2, tol=1e-3, max_iter=200, verbose=false)
    # 1-2. Endpoint steady states
    pre   = aiyagari_steady_state(p)
    p_new = @set p.A = A_new
    post  = aiyagari_steady_state(p_new; K_init=pre.K * A_new)

    hh      = aiyagari_household(p_new)
    K_path  = collect(range(pre.K, post.K; length=T))
    history = Float64[]

    for it in 1:max_iter
        # 3a. Build the env path from the current K guess
        env_path = [make_env(hh; aiyagari_prices(K_path[t], p_new)...) for t in 1:T]

        # 3b. One backward + one forward sweep, with the boundary conditions explicit
        (;V_path, Λ_path, moments_path) = solve_transition_given_env_path!(hh, env_path; Λ_0=pre.Λ, V_T=post.V)

        # 3c. Residual + damped update
        K_S = getproperty.(moments_path, :K_supplied)
        err = maximum(abs.(K_S .- K_path))
        push!(history, err)
        verbose && (it ≤ 5 || it % 5 == 0) &&
            @printf "  iter %3d: ‖K^S − K‖∞ = %.4e\n" it err

        err ≤ tol && return (; K_path, K_S, V_path, Λ_path, pre, post, history, iters=it)
        K_path .= (1 - update_speed) .* K_path .+ update_speed .* K_S
    end
    error("mit_shock_transition: did not converge")
end

T  = 100
tr = mit_shock_transition(p; A_new=1.05, T=T, verbose=true)
@printf "\nConverged in %d outer iters.\n" tr.iters
@printf "K_ss_pre  = %.4f\n" tr.pre.K
@printf "K_ss_post = %.4f\n" tr.post.K
@printf "K[1]   (impact) = %.4f\n" tr.K_path[1]
@printf "K[5]            = %.4f\n" tr.K_path[5]
@printf "K[20]           = %.4f\n" tr.K_path[20]
@printf "K[end] (≈post)  = %.4f\n" tr.K_path[end]

### 3.2 Manual version — what's inside `solve_transition_given_env_path!`

Same algorithm with every step exposed. Useful pedagogically: the
clean version *is* this loop, just packaged.

Three things this version makes visible:

- **Per-period chains** — `hh_path[t]` is one chain per period,
  sharing the Spec but each with its own Buffer. That way each
  period's backward result (the kernel) is preserved for the matching
  forward sweep, without redoing the work. A single chain would also
  work — the kernel cache would re-seat by re-running `backward!`
  whenever `forward!` saw a mismatched `V_end` — but that doubles the
  per-iteration work.
- **Boundary conditions as array endpoints** — `V_path[T+1] = post.V`
  and `Λ_path[1] = pre.Λ` sit at the array endpoints, taken directly
  from the steady-state returns. No accessor into chain internals.
- **`backward!` and `forward!` return their outputs** — `V_path[t]`
  flows out of `backward!`, `Λ_path[t+1]` flows out of `forward!`.
  Per-period moments come from `compute_moments(hh, Λ, env)`, with
  `Λ` passed in explicitly.


In [ ]:
function mit_shock_transition_manual(p::AiyagariParams; A_new=1.05, T=100, update_speed=0.2)
    # Endpoint steady states
    pre   = aiyagari_steady_state(p)
    p_new = @set p.A = A_new
    post  = aiyagari_steady_state(p_new; K_init=pre.K * A_new)

    # One chain per period — same spec, fresh buffer.
    hh_path = [aiyagari_household(p_new) for _ in 1:T]
    dims    = layout_size(aiyagari_layout(p_new))

    # V_path[t]   = continuation value at the start of period t
    # V_path[T+1] = post.V    (terminal boundary)
    # Λ_path[t]   = distribution at the start of period t
    # Λ_path[1]   = pre.Λ     (initial boundary)
    V_path = [zeros(Float64, dims...) for _ in 1:T+1]
    Λ_path = [zeros(Float64, dims...) for _ in 1:T+1]
    copyto!(V_path[T+1], post.V)
    copyto!(Λ_path[1],   pre.Λ)

    K_path  = collect(range(pre.K, post.K; length=T))
    history = Float64[]

    for it in 1:200
        env_path = [make_env(hh_path[t]; aiyagari_prices(K_path[t], p_new)...) for t in 1:T]

        # Backward sweep: V_t = backward!(V_{t+1}, env_t) on chain hh_path[t].
        # backward! returns V_start; copy it into V_path[t].
        for t in T:-1:1
            copyto!(V_path[t], backward!(hh_path[t], V_path[t+1], env_path[t]))
        end

        # Forward sweep: Λ_{t+1} = forward!(Λ_t) on chain hh_path[t].
        # The kernel cache is fresh (we just ran backward on this chain), so the
        # cheap forward call is correct by construction.
        K_S = zeros(T)
        for t in 1:T
            copyto!(Λ_path[t+1], forward!(hh_path[t], Λ_path[t]))
            K_S[t] = compute_moments(hh_path[t], Λ_path[t+1], env_path[t]).K_supplied
        end

        err = maximum(abs.(K_S .- K_path))
        push!(history, err)

        err ≤ 1e-3 && return (; K_path, K_S, V_path, Λ_path, pre, post, history, iters=it)
        K_path .= (1 - update_speed) .* K_path .+ update_speed .* K_S
    end
    error("mit_shock_transition_manual: did not converge")
end

tr_manual = mit_shock_transition_manual(p; A_new=1.05, T=T)
@printf "Manual converged in %d outer iters.\n" tr_manual.iters
@printf "‖K_path_clean − K_path_manual‖∞ = %.2e\n" maximum(abs.(tr.K_path .- tr_manual.K_path))

### 3.3 Capital IRF

The aggregate-capital path after the +5% TFP shock, with the pre-
and post-shock steady states marked as dashed horizontal lines.


In [ ]:
plot(1:T, tr.K_path; lw=2, label="K_t",
     xlabel="period t", ylabel="aggregate capital K",
     title="IRF: K to a +5% permanent TFP shock")
hline!([tr.pre.K];  color=:gray, linestyle=:dash, label="K_ss^pre")
hline!([tr.post.K]; color=:gray, linestyle=:dot,  label="K_ss^post")

### 3.4 Real-rate and wage paths

Read prices off the K path with `aiyagari_prices`, using the new
(post-shock) TFP parameter.


In [ ]:
prices_path = [aiyagari_prices(K, p_new) for K in tr.K_path]
r_path = getproperty.(prices_path, :r)
w_path = getproperty.(prices_path, :w)

plot(layout=(2, 1), size=(700, 500))
plot!(1:T, r_path; subplot=1, lw=2, label="r_t",
      xlabel="period", ylabel="r", title="IRF: real rate")
hline!([aiyagari_prices(tr.pre.K,  p     ).r]; subplot=1, color=:gray, linestyle=:dash, label="r_ss^pre")
hline!([aiyagari_prices(tr.post.K, p_new ).r]; subplot=1, color=:gray, linestyle=:dot,  label="r_ss^post")

plot!(1:T, w_path; subplot=2, lw=2, label="w_t",
      xlabel="period", ylabel="w", title="IRF: wage")
hline!([aiyagari_prices(tr.pre.K,  p     ).w]; subplot=2, color=:gray, linestyle=:dash, label="w_ss^pre")
hline!([aiyagari_prices(tr.post.K, p_new ).w]; subplot=2, color=:gray, linestyle=:dot,  label="w_ss^post")

### 3.5 Tatonnement residual history

Damped tatonnement drops the residual geometrically until it hits a
*discretization floor* at $\sim 2.5\times 10^{-3}$ on the baseline
calibration. The floor is the hard-`argmax` `ConsumptionSavingsStage`
policy flipping between adjacent grid cells as $K_t$ wobbles.
A smoothed (`LogitChoiceStage`-based) savings policy or a tighter
wealth grid would push the floor down.


In [ ]:
plot(1:length(tr.history), tr.history;
     yscale=:log10, lw=2, marker=:circle, markersize=3,
     xlabel="outer iteration", ylabel="‖K^S − K‖∞",
     label="residual", title="Damped tatonnement residual history")

### 3.6 Update-speed sweep

Update speed is a craft: too high oscillates, too low crawls. Re-run
the transition at $s \in \{0.1, 0.2, 0.4, 0.6\}$ and overlay the
residual histories.

**Expected.** $s = 0.6$ oscillates or fails to converge; $s = 0.1$
converges slowly; $s = 0.2$–$0.4$ is roughly the sweet spot.


In [ ]:
plt = plot(yscale=:log10, xlabel="outer iteration",
           ylabel="residual ‖K^S − K‖∞",
           title="Update-speed sweep")
for s in (0.1, 0.2, 0.4, 0.6)
    local r
    try
        r = mit_shock_transition(p; A_new=1.05, T=T, update_speed=s, tol=1e-3, max_iter=80)
    catch err
        # If a setting fails to converge inside max_iter, the function
        # errors; we still want to plot what residual history it had.
        @warn "s = $s did not converge; plotting partial history."
        continue
    end
    plot!(plt, 1:length(r.history), r.history; lw=2, label="s = $s")
end
plt